# 03 - Distributed Hyperparameter Tuning

This notebook demonstrates Snowflake's distributed hyperparameter optimization using `GridSearchCV` from `snowflake.ml.modeling.model_selection`.

**Key Concepts**:
- GridSearchCV parallelized across warehouse nodes
- No infrastructure setup required
- Results automatically tracked
- Best model registered as a new version

**Prerequisites**: Run `00_data_setup.ipynb` first.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print(f"Connected: {session.get_current_user()} | {session.get_current_role()}")

In [ ]:
# Load training data as a Snowpark DataFrame
# GridSearchCV works directly on Snowpark DataFrames (no .to_pandas() needed)
train_df = session.table("MLOPS_DEMO_DB.RAW.CUSTOMERS")

feature_cols = [
    "CREDIT_SCORE", "AGE", "TENURE", "BALANCE",
    "NUM_OF_PRODUCTS", "HAS_CR_CARD", "IS_ACTIVE_MEMBER", "ESTIMATED_SALARY"
]
label_cols = ["EXITED"]

print(f"Training data: {train_df.count()} rows, {len(feature_cols)} features")

In [ ]:
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.ml.modeling.model_selection import GridSearchCV

# Define the parameter grid to search
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 6, 9],
    "learning_rate": [0.01, 0.1, 0.3]
}

total_combinations = 1
for v in param_grid.values():
    total_combinations *= len(v)
print(f"Parameter grid: {total_combinations} combinations")
print(f"With 5-fold CV: {total_combinations * 5} total fits")

In [ ]:
# Run distributed GridSearchCV
# This is parallelized across the warehouse nodes automatically
grid_search = GridSearchCV(
    estimator=XGBClassifier(
        input_cols=feature_cols,
        label_cols=label_cols,
        output_cols=["PREDICTED_CHURN"]
    ),
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    input_cols=feature_cols,
    label_cols=label_cols,
    output_cols=["PREDICTED_CHURN"]
)

print("Starting distributed GridSearchCV...")
grid_search.fit(train_df)
print("GridSearchCV complete.")

In [ ]:
# Best parameters and score
print(f"Best F1 Score: {grid_search.to_sklearn().best_score_:.4f}")
print(f"\nBest Parameters:")
for param, value in grid_search.to_sklearn().best_params_.items():
    print(f"  {param}: {value}")

In [ ]:
# Show all CV results
import pandas as pd

cv_results = pd.DataFrame(grid_search.to_sklearn().cv_results_)
top_results = cv_results.nsmallest(5, "rank_test_score")[
    ["params", "mean_test_score", "std_test_score", "rank_test_score"]
]
print("Top 5 parameter combinations:")
print(top_results.to_string(index=False))

In [ ]:
# Register the best model as a new version in the registry
from snowflake.ml.registry import Registry

reg = Registry(
    session=session,
    database_name="MLOPS_DEMO_DB",
    schema_name="PIPELINES"
)

best_model = grid_search.to_sklearn().best_estimator_
best_score = grid_search.to_sklearn().best_score_

mv = reg.log_model(
    best_model,
    model_name="CHURN_PREDICTION_MODEL",
    version_name="v2_tuned",
    comment=f"Tuned XGBoost via GridSearchCV - best F1: {best_score:.4f}"
)

# Log the best metrics
mv.set_metric("f1_cv", round(best_score, 4))
for param, value in grid_search.to_sklearn().best_params_.items():
    mv.set_metric(f"best_{param}", value)

print(f"Registered: {mv.model_name} version {mv.version_name}")
print(f"Best F1 (CV): {best_score:.4f}")

## What to Show in SnowSight

Navigate to **AI/ML > Model Registry > CHURN_PREDICTION_MODEL** to see:
- Multiple versions (v1 from manual training, v2_tuned from HPO)
- Compare metrics across versions
- Best hyperparameters stored as metrics

**Key message**: Distributed HPO runs natively on Snowflake warehouses -- no external cluster, no Spark, no Ray setup required.

---

**Next**: `04_model_registry.ipynb` - Deep dive into Model Registry capabilities